# CLI (`craft-cdp` / `craft-pw`)

> Command-line entry point installed as `craft-cdp` (and `craft-pw`, shimmed by `dev/craft_pw.py`).

In [ ]:
#| default_exp cli

In [ ]:
#| export
import argparse, json, shutil, subprocess
from dataclasses import asdict
from urllib.parse import urlparse
from fastcore.utils import *

from craft_dev.auth import (
    DEFAULT_AUTH_FILE, DEFAULT_CHROME_PROFILE, DEFAULT_SOLVEIT_URL,
    default_target_url, import_cookies_json, login, save_cookie_header,)
from craft_dev.build import DEFAULT_CRAFT_NBS_DIR, DEFAULT_DIST_DIR, build_dist, sync_dist_to_root
from craft_dev.cdp import inspect_dialog, smoke_dialog
from craft_dev.tunnel import DEFAULT_RG_TOKEN_FILE, DEFAULT_TS_TAG, deploy_dist, ensure_local_tailscale, ensure_tunnel
from craft_dev.chrome_ext import DEFAULT_CHROME_EXT_DIR, build_chrome_ext

In [ ]:
#| export
def cmd_auth(args):
    auth_file = login(url=args.url, auth_file=Path(args.auth_file), profile_dir=Path(args.profile_dir), browser=args.browser,
        executable_path=args.executable_path, cdp_url=args.cdp_url, timeout=args.timeout)
    n = len(auth_file.read_json().get('cookies', []))
    print(f"Saved {n} cookies to {auth_file} (mode 0600).")
    return 0

_CLIPBOARD_CMDS = (('pbpaste',), ('wl-paste', '--no-newline'), ('xclip', '-selection', 'clipboard', '-o'), ('xsel', '--clipboard', '--output'))

def _read_clipboard():
    cmd = first(c for c in _CLIPBOARD_CMDS if shutil.which(c[0]))
    if not cmd: raise SystemExit("No clipboard utility found (tried pbpaste, wl-paste, xclip, xsel).")
    res = subprocess.run(cmd, capture_output=True, text=True)
    if res.returncode: raise SystemExit(f"{cmd[0]} failed: {res.stderr.strip()}")
    return res.stdout.strip()

def cmd_auth_cookie(args):
    if args.from_clipboard: header = _read_clipboard()
    elif sys.stdin.isatty():
        print(f"Paste the Cookie header (or Copy as cURL) for {args.url}, then press Enter:")
        header = sys.stdin.readline().strip()
        if header.lower() == 'cookie:': header = sys.stdin.readline().strip()
    else: header = sys.stdin.read().strip()

    if not header: raise SystemExit("No Cookie header provided.")
    cookies = save_cookie_header(header, args.url, auth_file=Path(args.auth_file))
    names = ', '.join(c['name'] for c in cookies)
    print(f"Saved {len(cookies)} cookie(s) ({names}) for {urlparse(args.url).hostname} to {args.auth_file} (mode 0600).")
    return 0

def cmd_auth_import(args):
    cookies = import_cookies_json(Path(args.cookies), auth_file=Path(args.auth_file))
    print(f"Imported {len(cookies)} cookies into {args.auth_file} (mode 0600).")
    return 0

def cmd_auth_status(args):
    report = inspect_dialog(url=args.url, auth_file=Path(args.auth_file), mode='live', wait_ms=1000,
        browser_name=args.browser, executable_path=args.executable_path)
    if report.authenticated:
        print(f"OK: authenticated at {report.final_url} (title: {report.title!r})")
        return 0
    print(f"EXPIRED: not authenticated at {report.final_url}. Run `craft-cdp auth` to refresh.")
    return 1

In [ ]:
#| export
def _print_inspect_report(report):
    print(f"URL:            {report.final_url}")
    print(f"Title:          {report.title}")
    print(f"Authenticated:  {report.authenticated}")
    print(f"Mode:           {report.mode} ({report.assets_dir})")
    print(f"Intercepted:    {len(report.intercepted)} CRAFT requests")
    for rec in report.intercepted: print(f"  - [{rec.action}] {rec.url}" + (f" -> {rec.local_path}" if rec.local_path else ''))
    if report.craft_health:
        h = report.craft_health
        active_globals = [k for k, v in h.get('globals', {}).items() if v is not None]
        print(f"Dialog cells:   {h.get('totalCells')} ({h.get('cellCounts')})")
        print(f"Active globals: {', '.join(active_globals) or '(none)'}")
        print(f"Hotkeys:        {h.get('hotkeyCount')}")
        print(f"CRAFT tags:     {h.get('craftTags')}")
    print(f"Page errors:    {len(report.page_errors)}")
    for err in report.page_errors: print(f"  ! {err}")
    print(f"Console errors: {len(report.console_errors)}")
    for err in report.console_errors: print(f"  ! {err}")
    if report.eval_result is not None: print(f"Eval result:    {json.dumps(report.eval_result, default=str)}")
    if report.screenshot: print(f"Screenshot:     {report.screenshot}")

def cmd_inspect(args):
    report = inspect_dialog(
        url=args.url, auth_file=Path(args.auth_file), mode=args.mode, assets_dir=Path(args.assets_dir).resolve() if args.assets_dir else None,
        wait_ms=args.wait_ms, eval_js=args.eval, screenshot=Path(args.screenshot) if args.screenshot else None,
        headed=args.headed, browser_name=args.browser, executable_path=args.executable_path, mobile=args.mobile)
    if args.json: print(json.dumps(asdict(report), indent=2, default=str))
    else: _print_inspect_report(report)
    return 0 if report.authenticated and not report.page_errors else 1

def cmd_smoke(args):
    checks = smoke_dialog(
        url=args.url, auth_file=Path(args.auth_file), mode=args.mode, assets_dir=Path(args.assets_dir).resolve() if args.assets_dir else None,
        wait_ms=args.wait_ms, screenshot_dir=Path(args.screenshot_dir) if args.screenshot_dir else None,
        headed=args.headed, browser_name=args.browser, executable_path=args.executable_path, mobile=args.mobile)
    for name, ok, detail in checks: print(f"[{'PASS' if ok else 'FAIL'}] {name}" + (f" — {detail}" if detail else ''))
    failed = [c for c in checks if not c[1]]
    print(f"\nSummary: {len(checks) - len(failed)}/{len(checks)} passed.")
    return 0 if not failed else 1

In [ ]:
#| export
def cmd_build(args):
    rep = build_dist(out_dir=Path(args.out), nbs_dir=Path(args.nbs_dir), check_js=not args.no_check)
    for name, info in sorted(rep.items()): print(f"  [{info['source']:6s}] {name} ({info['lines']} lines)")
    print(f"Built {len(rep)} assets into {args.out}")
    if args.sync:
        sync_res = sync_dist_to_root(dist_dir=Path(args.out))
        print(f"Synced {len(sync_res['copied'])} assets to repo root")
        if sync_res['orphaned']: print(f"Warning: orphaned root assets (not in CRAFT_ASSET_FILES): {', '.join(sync_res['orphaned'])}")
        ext_res = build_chrome_ext(check_js=not args.no_check)
        print(f"Built Chrome extension ({len(ext_res)} files) into {DEFAULT_CHROME_EXT_DIR}")
    return 0

def cmd_build_ext(args):
    rep = build_chrome_ext(out_dir=Path(args.out), check_js=not args.no_check)
    for name, sz in sorted(rep.items()): print(f"  [ext   ] {name} ({sz} bytes)")
    print(f"Built Chrome extension ({len(rep)} files) into {args.out}")
    return 0

def cmd_tunnel(args):
    if args.down:
        ensure_local_tailscale(down=True)
        print("Local Tailscale disconnected.")
        return 0
    res = ensure_tunnel(name=args.name, url=args.url, auth_file=Path(args.auth_file), rg_token_file=Path(args.rg_token_file), tag=args.tag,
        force=args.force)
    if res['status'] == 'needs-login':
        print(f"One-time Tailscale approval required for {res['name']}:\n  {res['auth_url']}\nApprove in your browser, then re-run `craft-cdp tunnel`.")
        return 0
    print(f"Tunnel [{res['status']}]: {res['name']} -> {res['url']} (registered in ~/.config/clikernel/gateways.toml)")
    return 0

def cmd_deploy(args):
    if not args.no_build: build_dist(out_dir=Path(args.dist_dir))
    res = deploy_dist(dist_dir=Path(args.dist_dir), name=args.name, url=args.url, auth_file=Path(args.auth_file))
    print(f"Deployed {res['files']} files ({res['bytes']:,} bytes) via {res['gateway']}")
    return 0

In [ ]:
#| export
def _add_dialog_args(p, saved_url):
    p.add_argument('--url', default=saved_url, required=not saved_url)
    p.add_argument('--mode', choices=['local', 'live', 'vanilla'], default='local')
    p.add_argument('--assets-dir', default=None)
    p.add_argument('--wait-ms', type=int, default=2000)
    p.add_argument('--headed', action='store_true')
    p.add_argument('--mobile', action='store_true')

def build_parser():
    parser = argparse.ArgumentParser(description="nbdev fastcdp auth, debugging, and smoke-testing tool for solve-it-crafts.")
    parser.add_argument('--auth-file', default=str(DEFAULT_AUTH_FILE))
    parser.add_argument('--browser', choices=['chrome', 'chromium', 'vivaldi'], default='chrome')
    parser.add_argument('--executable-path', default=None)
    sub = parser.add_subparsers(dest='command', required=True)
    saved_url = default_target_url()

    p_auth = sub.add_parser('auth', help="One-step Chrome login on the dedicated Solveit profile and auto-save session.")
    p_auth.add_argument('--url', default=os.environ.get('SOLVEIT_URL', DEFAULT_SOLVEIT_URL))
    p_auth.add_argument('--profile-dir', default=str(DEFAULT_CHROME_PROFILE))
    p_auth.add_argument('--timeout', type=int, default=300)
    p_auth.add_argument('--cdp-url', default=None)
    p_auth.set_defaults(func=cmd_auth)

    p_ck = sub.add_parser('auth-cookie', help="Save a Cookie header (or cURL) for `.solve.it.com`.")
    p_ck.add_argument('--url', default=DEFAULT_SOLVEIT_URL)
    p_ck.add_argument('--from-clipboard', action='store_true')
    p_ck.set_defaults(func=cmd_auth_cookie)

    p_imp = sub.add_parser('auth-import', help="Import exported browser cookies JSON into cookie state.")
    p_imp.add_argument('--cookies', required=True)
    p_imp.set_defaults(func=cmd_auth_import)

    p_stat = sub.add_parser('auth-status', help="Verify saved session state headlessly.")
    p_stat.add_argument('--url', default=DEFAULT_SOLVEIT_URL)
    p_stat.set_defaults(func=cmd_auth_status)

    p_ins = sub.add_parser('inspect', help="Inspect a Solveit dialog with local CRAFT overrides.")
    _add_dialog_args(p_ins, saved_url)
    p_ins.add_argument('--eval', dest='eval')
    p_ins.add_argument('--screenshot')
    p_ins.add_argument('--json', action='store_true')
    p_ins.set_defaults(func=cmd_inspect)

    p_smk = sub.add_parser('smoke', help="Run non-destructive UI smoke checks against a dialog.")
    _add_dialog_args(p_smk, saved_url)
    p_smk.add_argument('--screenshot-dir')
    p_smk.set_defaults(func=cmd_smoke)

    p_bld = sub.add_parser('build', help="Build dist/ assets from nbs/craft/ notebooks + unmigrated root files.")
    p_bld.add_argument('--out', default=str(DEFAULT_DIST_DIR))
    p_bld.add_argument('--nbs-dir', default=str(DEFAULT_CRAFT_NBS_DIR))
    p_bld.add_argument('--no-check', action='store_true')
    p_bld.add_argument('--sync', action='store_true', help='Copy built assets to repo root and flag orphaned root assets.')
    p_bld.set_defaults(func=cmd_build)

    p_ext = sub.add_parser('build-ext', help="Build the unpacked MV3 Chrome/Vivaldi extension in chrome-ext/.")
    p_ext.add_argument('--out', default=str(DEFAULT_CHROME_EXT_DIR))
    p_ext.add_argument('--no-check', action='store_true')
    p_ext.set_defaults(func=cmd_build_ext)

    p_tun = sub.add_parser('tunnel', help="Ensure private Tailscale + rustygate tunnel is up (or --down to disconnect).")
    p_tun.add_argument('--name', default='craft-test')
    p_tun.add_argument('--url', default=saved_url)
    p_tun.add_argument('--tag', default=DEFAULT_TS_TAG)
    p_tun.add_argument('--rg-token-file', default=str(DEFAULT_RG_TOKEN_FILE))
    p_tun.add_argument('--force', action='store_true')
    p_tun.add_argument('--down', action='store_true')
    p_tun.set_defaults(func=cmd_tunnel)

    p_dep = sub.add_parser('deploy', help="Build dist/ and deploy it atomically over the private rustygate tunnel.")
    p_dep.add_argument('--name', default='craft-test')
    p_dep.add_argument('--url', default=saved_url)
    p_dep.add_argument('--dist-dir', default=str(DEFAULT_DIST_DIR))
    p_dep.add_argument('--no-build', action='store_true')
    p_dep.set_defaults(func=cmd_deploy)
    return parser

def main():
    args = build_parser().parse_args()
    return args.func(args)

In [ ]:
from fastcore.test import test_eq

`build_parser` wires each subcommand to its handler with shared defaults for `inspect` and `smoke`.

In [ ]:
p = build_parser()
test_eq(p.parse_args(['auth']).func, cmd_auth)
ins = p.parse_args(['inspect', '--url', 'https://example.solve.it.com/dialog/1', '--json'])
test_eq((ins.func, ins.mode, ins.json), (cmd_inspect, 'local', True))
smk = p.parse_args(['smoke', '--url', 'https://example.solve.it.com/dialog/1', '--mode', 'vanilla'])
test_eq((smk.func, smk.mode), (cmd_smoke, 'vanilla'))
bld = p.parse_args(['build', '--sync'])
test_eq((bld.func, bld.sync), (cmd_build, True))
ext = p.parse_args(['build-ext'])
test_eq(ext.func, cmd_build_ext)